# Q33: PCA on Large Datasets
**Topic:** Classical-ml | **Difficulty:** Medium | **Time:** 20 min
**Sheet:** Grind75ML

---

## Question
Would you use PCA on large datasets or is there a better alternative?

## Detailed Answer

### PCA Limitation on Large Datasets
Standard PCA requires computing the full covariance matrix ($d \times d$) and its eigendecomposition:
- **Memory**: $O(d^2)$ for covariance matrix
- **Compute**: $O(d^3)$ for eigendecomposition
- For $d = 100{,}000$ features, covariance matrix needs ~80GB RAM

### Better Alternatives

| Method | Complexity | Accuracy | Best For |
|--------|-----------|----------|----------|
| **Randomized PCA** | O(nd·k) | Very close to exact | Large d, need top-k components |
| **Incremental PCA** | O(nd·k) per batch | Close to exact | Data doesn't fit in memory |
| **Truncated SVD** | O(nd·k) | Exact (sparse) | Sparse matrices (TF-IDF) |
| **Kernel PCA** | O(n²·k) | Non-linear | Non-linear dimensionality reduction |
| **t-SNE** | O(n² or n·log(n)) | N/A (non-parametric) | Visualization (2D/3D) |
| **UMAP** | O(n·log(n)) | N/A (non-parametric) | Visualization, faster than t-SNE |
| **Autoencoders** | Depends on arch | Can be non-linear | Very large, non-linear data |

### Recommendation
- **Use Randomized PCA** (sklearn's default when `svd_solver='randomized'`) — same quality, much faster
- **Use Incremental PCA** for out-of-memory data
- **Use Truncated SVD** for sparse data (don't center sparse matrices!)
- **Use UMAP** for visualization instead of t-SNE (faster, more scalable)

In [ ]:
import numpy as np
from sklearn.decomposition import PCA, IncrementalPCA, TruncatedSVD
from sklearn.datasets import make_classification
import time

# Generate large dataset
X, y = make_classification(n_samples=10000, n_features=500, n_informative=50, random_state=42)

# Compare methods
for name, model in [('PCA (full)', PCA(n_components=50, svd_solver='full')),
                     ('PCA (randomized)', PCA(n_components=50, svd_solver='randomized', random_state=42)),
                     ('Incremental PCA', IncrementalPCA(n_components=50, batch_size=1000)),
                     ('Truncated SVD', TruncatedSVD(n_components=50, random_state=42))]:
    start = time.time()
    X_transformed = model.fit_transform(X)
    elapsed = time.time() - start
    expl_var = sum(model.explained_variance_ratio_) if hasattr(model, 'explained_variance_ratio_') else 'N/A'
    print(f'{name:<25} Time: {elapsed:.3f}s  Shape: {X_transformed.shape}  Explained Var: {expl_var:.4f}' if isinstance(expl_var, float) else f'{name:<25} Time: {elapsed:.3f}s  Shape: {X_transformed.shape}')

## Key Takeaways
- **Randomized PCA** is the best default for large data — sklearn uses it automatically
- Never use full eigendecomposition PCA on datasets with >10K features
- For sparse data (NLP), use **TruncatedSVD** — PCA centers data and destroys sparsity
- UMAP has largely replaced t-SNE for visualization due to speed and scalability